Shortly, what i want to implement with this code is a gemmav4 which as released in 2 april 2026 and I want use it to annotated the rest of my rows from car advertisments dataset.
I did this architecture :

1. `teacher` model proposes labels
2. `verifier` model checks the proposal
3. `router` decides `accept`, `send_to_review`, or `ask_human`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT_ROOT = "/content/drive/MyDrive/Gliner-Work.Dauphine"

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Data loading

In [2]:
from pathlib import Path


root = Path("/Users/raresolteanu/Desktop/Gliner-Work.Dauphine")
DATASET_PATH = root / "/content/annotation_working_master_human_2100_seed.csv"
OUTPUT_DIR = root / "gemma_review_pipeline" / "outputs"

TEXT_COLUMNS = [
    "Script",
    "Incrustation",
    "Visuel",
    "Titre",
    "MotsClés",
    "Thème",
    "Signature",
]

METADATA_COLUMNS = [
    "year",
    "Month",
    "Marque",
    "Produit",
    "Variété",
    "Electric",
    "Hybrid",
    "Support",
    "Format",
    "Agence",
    "length",
]

VALID_LABELS = [
    "Comfort/Practicality",
    "Design/Emotion",
    "Ecology/Sustainability",
    "Electrification",
    "Innovation/Technology",
    "Luxury/Status",
    "Performance",
    "Price/Value",
    "Safety/Reliability",
]

ROUTER_ACCEPT_THRESHOLD = 0.78
ROUTER_REVIEW_THRESHOLD = 0.55
ROUTER_HUMAN_THRESHOLD = 0.35

#GEMMA_MODEL_ID= "google/gemma-4-E2B-it" #If you run it locally, I suggest run thism if the data is over 1000 to avoid kernel crash, but it has the problem with weakest reasoning and weakest label accuracy
#GEMMA_MODEL_ID= "google/gemma-4-E4B-it" #balance of speed and quality, easier to fine-tune, it can work on better laptops
#GEMMA_MODEL_ID = "google/gemma-4-26B-A4B-it" #speed/quality tradeoff, MoE model with only about 3.8B active parameters at inference, but memory pressure
GEMMA_MODEL_ID = "google/gemma-4-31B-it" #best quality, strongest reasoning, best for maximum performance, but slowest, heaviest, most expensive,


imports for the next code

In [3]:
from typing import Iterable
import pandas as pd

from gemma_review_pipeline.heuristics import pick_top_two, score_labels
from gemma_review_pipeline.schemas import (
    AnnotationPrediction,
    RouterDecision,
    VerifierDecision,
)

Firstly, we implemet a normalise function

In [4]:
def normalize_text(value: object) -> str: #here we take any value kind and we return string
    if pd.isna(value): #here we check if there is any missing value, such as NaN or None
        return "" #if the value is missing it returns a empthy string
    return " ".join(str(value).split()).strip() # the values is transformed into string, than it breaks into words and remove spaces and then it put all together again and we remove the spaces from the beginning and end


then we load the rows

In [5]:
def load_rows(limit: int | None = None, review_status: str = "needs_label") -> pd.DataFrame: # we return a dataframe and two optional inputs the limit and review_status, default being needs_label
    df = pd.read_csv(DATASET_PATH)
    if review_status:
        df = df[df["Review_Status"] == review_status].copy() #keeps where the rows match Review_Status and copy is to avoid pandas warning
    df["row_id"] = pd.to_numeric(df["row_id"], errors="coerce").astype("Int64") # we clean the row_id and we change the invalid values to NaN and Int64 to not allow missing values
    if limit is not None:
        df = df.head(limit).copy()
    return df

We build here the build record text function

In [6]:
def build_record_text(row: pd.Series, columns: Iterable[str] | None = None) -> str: #it takes one row df a list of columns to the return
    columns = list(columns or TEXT_COLUMNS)# use columns if provided if not use TEXT_COLUMNS
    parts: list[str] = [] #create an empthy list
    for column in columns:
        text = normalize_text(row.get(column, ""))# get the value if the column existsm otherwise empthy string and clean it
        if text:
            parts.append(f"{column}: {text}")# add the column name and its text
    return "\n".join(parts) # join all of it in one string separated by line breaks

then we build also the meta data function for text this is the exact same function, but it is for METADATA_COLUMNS

In [7]:
def build_metadata_text(row: pd.Series, columns: Iterable[str] | None = None) -> str:
    columns = list(columns or METADATA_COLUMNS)
    parts: list[str] = []
    for column in columns:
        text = normalize_text(row.get(column, ""))
        if text:
            parts.append(f"{column}: {text}")
    return "\n".join(parts)

 First annotator, which is a dummy teacher.( it return JSON with:
- main_benefit
- secondary_benefit
- confidence
- alt_label
- reason)



The used labels
Valid labels:
Comfort/Practicality, Design/Emotion, Ecology/Sustainability, Electrification,
Innovation/Technology, Luxury/Status, Performance, Price/Value, Safety/Reliability


The function for the teacher prompt

In [8]:
def build_teacher_prompt(row: pd.Series) -> str: #transform the row into pandas and then it turn it in string
    return f"""You are an annotation model for automotive ads.

Metadata:
{build_metadata_text(row)}

Content:
{build_record_text(row)}
"""



the anotation prediction function

In [9]:
def dummy_teacher_predict(row: pd.Series) -> AnnotationPrediction: #takes one row  from df and transform it into a annotation
    text = build_record_text(row) #combines Script, Incrustation etc. into strings
    metadata = {key: row.get(key) for key in row.index} # a dict with all values from rows, this is used to so score fun can look at metadata
    scores = score_labels(text, metadata) # this give to the score fun the text and metadata and it return a score for each label possible
    top, second = pick_top_two(scores) # select the top 2 labels
    total = sum(max(value, 0.0) for value in scores.values()) or 1.0 #add all pos values and prevents neg from reducing the total and or 1 is to avoid the division by 0
    confidence = min(0.95, max(0.2, top[1] / total + 0.35)) #cofidence scorem it measures how strong the label is compared with others and with min we make sure the confi not too high and wth max not too low
    secondary = second[0] if second[0] and second[0] != top[0] else "" # this choose the second label as secondary_benefit, if it exists and is not like the previous one, otherwise it leaves it empty
    reason = f"keyword score main={top[0]}:{top[1]:.1f}; alt={second[0]}:{second[1]:.1f}"
    return AnnotationPrediction(
        row_id=int(row["row_id"]), #rows id as int
        main_benefit=top[0],
        secondary_benefit=secondary,
        confidence=round(confidence, 4), #save confidence rounded to 4 deci
        alt_label=second[0], #the alt label
        reason=reason, #as str
        model_name="dummy_teacher", # prediction came from teh teacher
    )

How many rows to annotate function

In [10]:
def annotate_rows(limit: int = 20) -> pd.DataFrame: #a fun for how many rows to annotate, default is 20
    rows = load_rows(limit=limit, review_status="needs_label") #load the CSV an donly keeps rows where Review_Status == "needs_label"
    predictions = [dummy_teacher_predict(row).to_dict() for _, row in rows.iterrows()] #it loops through every rowws and makes a pred and convert it in dictm the first '_' is row index
    return pd.DataFrame(predictions)

In [11]:
def main() -> None: #the entry point of a script and it return None
    parser = argparse.ArgumentParser() #create an argument parser, this is used with scope of running it from terminal and pass options
    parser.add_argument("--limit", type=int, default=20) #adds one optional arg limit, it needs to be int and the default is 20
    args = parser.parse_args() #reads the arg and stores them in args

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True) #creates a output folderm create missing parent folders and do not crash if it already exists
    output_path = OUTPUT_DIR / "teacher_predictions.csv" # a full path
    predictions = annotate_rows(limit=args.limit) # runs theannotate_rows() fun on how many rows are req
    predictions.to_csv(output_path, index=False)# save df to csv file and it does not save row nr
    print("teacher model id placeholder:", GEMMA_MODEL_ID) #model id
    print("wrote", output_path) #where the file is saved
    print(predictions.head().to_string(index=False)) #first few rows


  Second model, it uses a narrower evidence slice.

In [12]:
def verifier_predict(row: pd.Series, teacher_main: str) -> VerifierDecision: # takes a row and the main label proposed by teacher and it returns a VerifierDecision
    text = "\n".join(
        part
        for part in [
            str(row.get("Incrustation", "")),
            str(row.get("Visuel", "")),
            str(row.get("Titre", ""))
        ] #gets the values and transform them in stringsm only these columnsm since verifier is using a narrower evidence than teacher
        if part and part != "nan"
    ) #keeps only non-empty or remove it
    #the rest of the function uses the same codes as beforem but adapted to the Verifier and a little finetuned
    metadata = {key: row.get(key) for key in row.index}
    scores = score_labels(text, metadata)
    top, second = pick_top_two(scores)
    total = sum(max(value, 0.0) for value in scores.values()) or 1.0
    confidence = min(0.95, max(0.15, top[1] / total + 0.25))
    agrees = top[0] == teacher_main #this is verifies it to be equal with the teacher prediction
    reason = "agreement" if agrees else f"teacher={teacher_main}, verifier={top[0]}, alt={second[0]}"
    return VerifierDecision(
        row_id=int(row["row_id"]), #row id
        verifier_main_benefit=top[0],
        verifier_confidence=round(confidence, 4),
        agrees_with_teacher=agrees, #stores if it agrees with teacher or not
        disagreement_reason=reason, # explanation for the above
    )


In [13]:
def run_verifier(limit: int = 20) -> pd.DataFrame:
    rows = load_rows(limit=limit, review_status="needs_label")
    teacher = pd.read_csv(OUTPUT_DIR / "teacher_predictions.csv") #load teh teacher predictions
    merged = rows.merge(teacher[["row_id", "main_benefit"]], on="row_id", how="inner") #Combines the original dataset rows with the teacher predictions
    decisions = [
        verifier_predict(row, row["main_benefit"]).to_dict() #the second arg is the teacher label for the same row
        for _, row in merged.iterrows()
    ]
    return pd.DataFrame(decisions)

The same function as in the teacher, but adapted to this

In [14]:
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--limit", type=int, default=20)
    args = parser.parse_args()

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / "verifier_decisions.csv"
    decisions = run_verifier(limit=args.limit)
    decisions.to_csv(output_path, index=False)
    print("wrote", output_path)
    print(decisions.head().to_string(index=False))

   this model decides whether to accept, review, or ask a human.

In [15]:
def route_row(row: pd.Series) -> RouterDecision:#takes one row and gives a RouterDecision object.
    teacher_conf = float(row["confidence"]) #reads the teacher confi and make it float
    verifier_conf = float(row["verifier_confidence"]) #the same but for verifier
    agrees = bool(row["agrees_with_teacher"]) #check if the two agrees with eachother
    route_score = (teacher_conf + verifier_conf) / 2.0 #computes the avg confi between the two, an overall routing score
 # first decison rule if teacher and veriefier agree and both teacher and verifier confi exced the treshold, then trust the predi
    if agrees and teacher_conf >= ROUTER_ACCEPT_THRESHOLD and verifier_conf >= ROUTER_REVIEW_THRESHOLD:
        action = "accept" #mark the row as accepted automatically
        reason = "teacher and verifier agree with strong confidence" #explanation
#second decision rule if teacher and verifier disagree and the average score is low and teacher confi low the the row is risky
    elif (not agrees and route_score < ROUTER_REVIEW_THRESHOLD) or teacher_conf <= ROUTER_HUMAN_THRESHOLD:
        action = "ask_human" #what the model should do
        reason = "low confidence or verifier disagreement" #explanation
#third decision rule if the row is not clearly safe and not clearly bad, it is in the middle.
    else:
        action = "send_to_review" #what the model should do
        reason = "uncertain case worth a second pass" # explanation

    return RouterDecision(
        row_id=int(row["row_id"]), #save row id
        route_action=action, #save the chosen action
        route_score=round(route_score, 4), #save the avg confi
        route_reason=reason, #save the explanation
    )

In [16]:
def run_router() -> pd.DataFrame:
    teacher = pd.read_csv(OUTPUT_DIR / "teacher_predictions.csv")
    verifier = pd.read_csv(OUTPUT_DIR / "verifier_decisions.csv")
    merged = teacher.merge(verifier, on="row_id", how="inner")
    actions = [route_row(row).to_dict() for _, row in merged.iterrows()]
    return pd.DataFrame(actions)

In [17]:
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.parse_args()

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / "router_actions.csv"
    actions = run_router()
    actions.to_csv(output_path, index=False)
    print("wrote", output_path)
    print(actions["route_action"].value_counts().to_string())

  We do a check with a small sample.


In [18]:
def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--limit", type=int, default=20)
    args = parser.parse_args()

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    teacher = annotate_rows(limit=args.limit)
    teacher.to_csv(OUTPUT_DIR / "teacher_predictions.csv", index=False)

    verifier = run_verifier(limit=args.limit)
    verifier.to_csv(OUTPUT_DIR / "verifier_decisions.csv", index=False)

    router = run_router()
    router.to_csv(OUTPUT_DIR / "router_actions.csv", index=False)

    print("demo complete")
    print("teacher rows:", len(teacher))
    print("verifier rows:", len(verifier))
    print("router rows:", len(router))
    print(router.head().to_string(index=False))


In [19]:
limit = 20

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

teacher = annotate_rows(limit=limit)
teacher.to_csv(OUTPUT_DIR / "teacher_predictions.csv", index=False)

verifier = run_verifier(limit=limit)
verifier.to_csv(OUTPUT_DIR / "verifier_decisions.csv", index=False)

router = run_router()
router.to_csv(OUTPUT_DIR / "router_actions.csv", index=False)

print("demo complete")
print(router.head().to_string(index=False))


demo complete
 row_id   route_action  route_score                                      route_reason
   2100 send_to_review       0.8143                uncertain case worth a second pass
   2101 send_to_review       0.7040                uncertain case worth a second pass
   2102         accept       0.9500 teacher and verifier agree with strong confidence
   2103         accept       0.8357 teacher and verifier agree with strong confidence
   2104         accept       0.7722 teacher and verifier agree with strong confidence
